In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [3]:
# ============================================================
# IND + Markov snow model
#   p(y_{s,t} = 1 | y_{s,t-1})
#   logit p = beta_s^T X_t + delta * y_{s,t-1}
# ============================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import coo_matrix, diags, bmat, hstack, csr_matrix
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr

# ============================================================
# Load data
# ============================================================

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

df1 = snow.drop(index=no_nbs)
df2 = snow.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

# y: S x T (binary)
y = all_y.iloc[:, 2:].to_numpy()
S, TT = y.shape
period = 52

print(f"S = {S}, T = {TT}")

# ============================================================
# Build dataset: all (s,t), t = 2,...,T
# ============================================================

# indices
row_idx, time_idx = np.meshgrid(
    np.arange(S),
    np.arange(1, TT),   # t = 1,...,T-1  (0-based)
    indexing="ij"
)

row_idx = row_idx.ravel()
time_idx = time_idx.ravel()
N = len(row_idx)

# response and Markov covariate
y_curr = y[row_idx, time_idx]        # y_{s,t}
y_prev = y[row_idx, time_idx - 1]    # y_{s,t-1}

kappa = y_curr - 0.5                 # PG trick

t = time_idx + 1                     # time index (1-based)

# ============================================================
# Covariates
#   - site-specific: intercept, cos, sin, t
#   - global: y_{t-1}
# ============================================================

X_local = np.column_stack([
    np.ones(N),
    np.cos(2*np.pi*t / period),
    np.sin(2*np.pi*t / period),
    t
])

X_markov = y_prev.reshape(-1, 1)

K = X_local.shape[1]      # 4
theta_dim = K * S + 1     # +1 for delta

print("N =", N)
print("theta_dim =", theta_dim)

# ============================================================
# Build design matrix
# ============================================================

rows, cols, vals = [], [], []

for i in tqdm(range(N), desc="Building site-specific design"):
    s = row_idx[i]
    for k in range(K):
        rows.append(i)
        cols.append(s + k * S)
        vals.append(X_local[i, k])

X_local_mat = coo_matrix(
    (vals, (rows, cols)),
    shape=(N, K * S)
).tocsr()

X_markov_mat = csr_matrix(X_markov)

design_mat = hstack([X_local_mat, X_markov_mat], format="csr")

# ============================================================
# Prior precision
#   beta_s ~ N(0, I)
#   delta  ~ N(0, 1)
# ============================================================

prior_prec = 1.0

blocks = []
for i in range(K):
    row = []
    for j in range(K):
        row.append(prior_prec * diags(np.ones(S)) if i == j else None)
    blocks.append(row)

Q_local = bmat(blocks, format="csr")
Q_markov = csr_matrix([[prior_prec]])

Q_prior = bmat(
    [[Q_local, None],
     [None,   Q_markov]],
    format="csr"
)

# ============================================================
# MCMC settings
# ============================================================

burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
curr_theta = np.zeros(theta_dim)

save_idx = 0

# ============================================================
# PG–Gibbs sampler
# ============================================================

for it in tqdm(range(total_iters), desc="MCMC (IND + Markov)"):

    # 1. PG augmentation
    phi = design_mat @ curr_theta
    omega = random_polyagamma(1, phi, size=N)

    # 2. Posterior precision
    xtOmega = design_mat.T.multiply(omega)
    post_prec = xtOmega @ design_mat + Q_prior

    factor = cholesky(post_prec)

    # 3. Sample theta
    mu = factor.solve_A(design_mat.T @ kappa)
    curr_theta = mu + factor.solve_A(np.random.randn(theta_dim))

    # 4. Save
    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:, save_idx] = curr_theta
        save_idx += 1
        if save_idx >= tot_save:
            break

# ============================================================
# Save output
# ============================================================

np.savez_compressed(
    r"D:\77\Research\temp\snow\ind_markov.npz",
    all_theta=all_theta
)

print("Done. Saved to ind_markov.npz")


S = 1618, T = 2704
N = 4373454
theta_dim = 6473


MCMC (IND + Markov):   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_73284\3989241969.py:156: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec)
MCMC (IND + Markov): 100%|█████████▉| 5995/6000 [1:40:01<00:05,  1.00s/it]


Done. Saved to ind_markov.npz
